In [1]:
import pandas as pd
import numpy as np
import pickle

In [ ]:
"""

countsdf will give summary about how many variants per kb are occurring in region with actiuvation domain vs non-activation domain

effect with high: high consequences of mutation (PTC), frame shift mutations
effect with moderate: nonsynonymous mutation
effect with low: sysnonynmous muattions

Full documentations: https://pcingola.github.io/SnpEff/snpeff/outputsummary/

"""

In [2]:
##less stringest threshold

variantdata=pd.read_csv('../sorghumdata/annotated_sorghum_TFs_variant_effects.csv')
TFs=pd.read_csv('../sorghumdata/Sorghum_ActivityAnnotated.csv')
TFs['protein_ID']=TFs['protein_ID'].str.replace('.p', '')
TFs['trancript']=TFs['protein_ID']

activators=set(TFs[TFs['Action']=='Activator']['trancript'])
variantdata.rename(columns={'gene': 'gene_ID'}, inplace=True)

file_path = '../sorghumdata/sbi_all_preds.pkl' # Replace with your file's actual path
with open(file_path, 'rb') as file:
    data = pickle.load(file)

data=data[['gene_ID','protein_ID','family','seq', 'activity_avg']]
data['protein_ID']=data['protein_ID'].str.replace('.p', '')
data['trancript']=data['protein_ID']

transcript_nomatch=set()

transcripts_activator=set()
for _, row in variantdata.iterrows():
    
    transcript=row['transcript']

    if not transcript in activators: continue

    arr=data.loc[data['trancript'] == transcript, 'activity_avg'].iloc[0]

    variantpos=int(row['protein_position'])-1

    if (len(arr)!=int(row['protein_length'])):
        # variantdata.at[_, 'activity']='None'
        transcript_nomatch.add(transcript)
        continue
    if int(row['protein_position'])>len(arr):
        print(transcript, int(row['protein_position']), len(arr)) ##because of stop codon not included
        continue

    transcripts_activator.add(transcript)

    if arr[variantpos]>1:
        variantdata.at[_, 'activity']='AD'
    else:
        variantdata.at[_, 'activity']='non-AD'
variantdata = variantdata[variantdata['activity'].isin(['AD', 'non-AD'])]
activatorlen=0
nonactivatorlen=0

for ta in transcripts_activator:
    arr=data.loc[data['trancript'] == ta, 'activity_avg'].iloc[0]

    for arr_num in arr:
        if arr_num>1:
            activatorlen+=1
        else:
            nonactivatorlen+=1

print(activatorlen, nonactivatorlen)

# counts = variantdata.groupby(['activity', 'effect']).size()
# # countsdf=pd.DataFrame(counts)

# # countsdf.columns=['effect', 'total']
countsdf = (
    variantdata
      .groupby(['activity', 'effect'])
      .size()
      .reset_index(name='total')
)
countsdf['length'] = np.where(
    countsdf['activity'] == 'AD',
    activatorlen,
    nonactivatorlen
)

countsdf['variantsperkb']=(countsdf['total']/countsdf['length'])*1000
countsdf

Sobic.003G337500.1 237 236
Sobic.004G207200.1 339 338
Sobic.004G219800.1 259 258
Sobic.006G093700.1 635 634
Sobic.007G090100.1 286 285
Sobic.008G060300.1 335 334
Sobic.009G039700.1 357 356
Sobic.010G273700.1 387 386
44701 294053


,activity,effect,total,length,perkb
0,AD,HIGH,433,44701,9.686584
1,AD,LOW,498,44701,11.140690
2,AD,MODERATE,569,44701,12.729022
3,non-AD,HIGH,3397,294053,11.552339
4,non-AD,LOW,2942,294053,10.004999
5,non-AD,MODERATE,3606,294053,12.263095


In [3]:
## more stringent threshold
variantdata=pd.read_csv('../sorghumdata/annotated_sorghum_TFs_variant_effects.csv')
TFs=pd.read_csv('../sorghumdata/Sorghum_ActivityAnnotated.csv')
TFs['protein_ID']=TFs['protein_ID'].str.replace('.p', '')
TFs['trancript']=TFs['protein_ID']

activators=set(TFs[TFs['Action']=='Activator']['trancript'])
variantdata.rename(columns={'gene': 'gene_ID'}, inplace=True)

file_path = '../sorghumdata/sbi_all_preds.pkl' # Replace with your file's actual path
with open(file_path, 'rb') as file:
    data = pickle.load(file)

data=data[['gene_ID','protein_ID','family','seq', 'activity_avg']]
data['protein_ID']=data['protein_ID'].str.replace('.p', '')
data['trancript']=data['protein_ID']

LOW=-1
HIGH=0.5

transcript_nomatch=set()

transcripts_activator=set()
for _, row in variantdata.iterrows():
    
    transcript=row['transcript']

    if not transcript in activators: continue

    arr=data.loc[data['trancript'] == transcript, 'activity_avg'].iloc[0]

    variantpos=int(row['protein_position'])-1

    if (len(arr)!=int(row['protein_length'])):
        # variantdata.at[_, 'activity']='None'
        transcript_nomatch.add(transcript)
        continue
    if int(row['protein_position'])>len(arr):
        print(transcript, int(row['protein_position']), len(arr))
        continue

    transcripts_activator.add(transcript)

    if arr[variantpos]>3:
        variantdata.at[_, 'activity']='AD'
    elif (arr[variantpos]>=LOW) & (arr[variantpos]<=HIGH):
        variantdata.at[_, 'activity']='non-AD'

        
variantdata = variantdata[variantdata['activity'].isin(['AD', 'non-AD'])]

activatorlen=0
nonactivatorlen=0

for ta in transcripts_activator:
    arr=data.loc[data['trancript'] == ta, 'activity_avg'].iloc[0]

    for arr_num in arr:
        if arr_num>3:
            activatorlen+=1
        elif (arr_num >= LOW) & (arr_num <= HIGH):
            nonactivatorlen+=1

print(activatorlen, nonactivatorlen)

countsdf = (
    variantdata
      .groupby(['activity', 'effect'])
      .size()
      .reset_index(name='total')
)
countsdf['length'] = np.where(
    countsdf['activity'] == 'AD',
    activatorlen,
    nonactivatorlen
)

countsdf['variantsperkb']=(countsdf['total']/countsdf['length'])*1000
countsdf

Sobic.003G337500.1 237 236
Sobic.004G207200.1 339 338
Sobic.004G219800.1 259 258
Sobic.006G093700.1 635 634
Sobic.007G090100.1 286 285
Sobic.008G060300.1 335 334
Sobic.009G039700.1 357 356
Sobic.010G273700.1 387 386
11104 256411


,activity,effect,total,length,perkb
0,AD,HIGH,109,11104,9.816282
1,AD,LOW,115,11104,10.356628
2,AD,MODERATE,124,11104,11.167147
3,non-AD,HIGH,2935,256411,11.446467
4,non-AD,LOW,2577,256411,10.050271
5,non-AD,MODERATE,3125,256411,12.187465
